# 📊 EDA — Engagement Score V2: Smoothed IDF

**Metodologi:** Invers Weight dengan alpha-smoothing per platform & per faktor

$$IDF_{p,f} = \log_2 \left( \frac{N_p + \alpha}{DF_{p,f} + \alpha} \right)$$

| Variabel | Definisi |
|---|---|
| **N_p** | Total post di platform *p* |
| **DF_{p,f}** | Jumlah post di platform *p* yang punya engagement faktor *f* > 0 |
| **α** | Konstanta smoothing (default=1.0) |

**Formula Akhir:**
$$\text{Engagement Score} = \sum (\text{nilai\_faktor} \times \text{IDF}_{p,f\_norm})$$

| Platform | Faktor |
|---|---|
| Facebook | Shares, Likes, Reply |
| Twitter | Shares(retweet), Views, Reply, Likes, Retweet |
| Instagram | Shares, Reply, Likes, **Repost** (via reshare_count) |
| TikTok | Shares, Views, Reply, Likes |
| YouTube | Shares, Views, Reply/Comment, Likes+Dislikes |
| Threads | Shares, Reply, Repost, Quote, Likes |

In [ ]:
import json, glob, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 6)
sns.set_theme(style='darkgrid', palette='muted')
COLORS = {
    'facebook':'#1877F2','twitter':'#1DA1F2','instagram':'#E1306C',
    'tiktok':'#555555','youtube':'#FF0000','threads':'#444444'
}
BASE_PATH = Path(r'D:/SPECTRA/Riset_enggagement/top100_raw_20260805')
PLATFORMS = ['facebook','twitter','instagram','tiktok','youtube','threads']
ALPHA = 1.0  # smoothing constant
print(f'✅ Setup selesai | ALPHA={ALPHA}')

## 1️⃣ Load & Parsing Data

In [ ]:
def safe(d, *keys, default=0):
    for k in keys:
        if isinstance(d, dict): d = d.get(k, {})
        else: return default
    return d if isinstance(d,(int,float)) else default

def extract_facebook(r):
    return {'post_id': r.get('id','?'),
            'likes': safe(r,'feedback','reaction_count','count'),
            'reply': safe(r,'feedback','comments_count_reduced','count'),
            'shares': safe(r,'feedback','share_count','count')}

def extract_twitter(r):
    lg = r.get('legacy',{})
    return {'post_id': r.get('rest_id','?'),
            'likes': lg.get('favorite_count',0) or 0,
            'reply': lg.get('reply_count',0) or 0,
            'retweet': lg.get('retweet_count',0) or 0,
            'views': int(r.get('views',{}).get('count',0) or 0)}

def extract_instagram(r):
    # Repost diambil dari reshare_count.
    # (Catatan: seringkali bernilai null/0 di raw data)
    return {'post_id': r.get('pk',r.get('id','?')),
            'likes': r.get('like_count',0) or 0,
            'reply': r.get('comment_count',0) or 0,
            'shares': 0,  # Instagram shares tidak tersedia di raw data
            'repost': r.get('reshare_count',0) or 0}

def extract_tiktok(r):
    return {'post_id': r.get('id','?'),
            'likes': r.get('digg_count',0) or 0,
            'shares': r.get('share_count',0) or 0,
            'reply': r.get('comment_count',0) or 0,
            'views': r.get('play_count',0) or 0}

def extract_youtube(r):
    return {'post_id': r.get('id',r.get('url','?')),
            'likes': r.get('likes',0) or 0,
            'reply': r.get('commentCount', r.get('comments',0)) or 0,
            'views': r.get('views',0) or 0,
            'shares': 0}

def extract_threads(r):
    tpi = r.get('text_post_app_info',{})
    return {'post_id': r.get('pk',r.get('id','?')),
            'likes': r.get('like_count',0) or 0,
            'reply': tpi.get('direct_reply_count',0) or 0,
            'repost': tpi.get('repost_count',0) or 0,
            'quote': tpi.get('quote_count',0) or 0,
            'shares': tpi.get('reshare_count',0) or 0}

EXTRACTORS = {
    'facebook':extract_facebook, 'twitter':extract_twitter,
    'instagram':extract_instagram, 'tiktok':extract_tiktok,
    'youtube':extract_youtube, 'threads':extract_threads
}

def load_platform(plat):
    records = []
    ext = EXTRACTORS[plat]
    for fp in glob.glob(str(BASE_PATH/plat/'*.json')):
        try:
            with open(fp,encoding='utf-8') as f:
                raw = json.load(f)
            items = raw if isinstance(raw,list) else [raw]
            for item in items:
                records.append(ext(item))
        except: pass
    df = pd.DataFrame(records)
    df['platform'] = plat
    return df

dfs = {p: load_platform(p) for p in PLATFORMS}
for p,df in dfs.items():
    print(f'{p:12s} → {len(df):4d} posts | cols: {[c for c in df.columns if c not in ["post_id","platform"]]}')

## 2️⃣ Statistik Deskriptif per Platform

In [ ]:
for p,df in dfs.items():
    eng = [c for c in df.columns if c not in ['post_id','platform']]
    print(f'\n📌 {p.upper()}')
    print(df[eng].describe().round(2).to_string())
    print('-'*60)

## 3️⃣ Hitung DF_{p,f}: Jumlah Post dengan Engagement > 0

In [ ]:
print('='*65)
print('📐 DF_{p,f} — Jumlah post dengan nilai engagement faktor > 0')
print('='*65)

df_stats = {}
for plat, df in dfs.items():
    eng = [c for c in df.columns if c not in ['post_id','platform']]
    N_p = len(df)
    stat = {'N_p': N_p}
    print(f'\n🔵 {plat.upper()} | N_p={N_p}')
    for col in eng:
        df_pf = (df[col] > 0).sum()
        pct = df_pf/N_p*100
        stat[col] = df_pf
        print(f'   DF[{col:12s}] = {df_pf:4d} / {N_p} ({pct:5.1f}%)')
    df_stats[plat] = stat

## 4️⃣ Perhitungan IDF V2 dengan Smoothing

In [ ]:
results_v2 = {}

for plat, df in dfs.items():
    eng = [c for c in df.columns if c not in ['post_id','platform']]
    N_p = df_stats[plat]['N_p']
    
    idf_raw, idf_norm = {}, {}
    for col in eng:
        DF_pf = df_stats[plat].get(col, 0)
        # ✅ V2: IDF dengan smoothing alpha → tidak ada division-by-zero
        idf = np.log2((N_p + ALPHA) / (DF_pf + ALPHA))
        idf_raw[col] = idf
    
    # Normalisasi ke [0,1]
    vals = np.array(list(idf_raw.values()))
    vmin, vmax = vals.min(), vals.max()
    for col, v in idf_raw.items():
        idf_norm[col] = (v - vmin)/(vmax - vmin) if vmax != vmin else 1/len(idf_raw)
    
    # Engagement Score
    score = pd.Series(0.0, index=df.index)
    for col, w in idf_norm.items():
        score += df[col].fillna(0) * w
    
    df_out = df.copy()
    df_out['engagement_score_v2'] = score
    results_v2[plat] = {'df': df_out, 'idf_raw': idf_raw, 'idf_norm': idf_norm, 'N_p': N_p}

print('='*65)
print('📐 IDF WEIGHTS V2 (Smoothed) — Raw & Normalized')
print('='*65)
for plat, res in results_v2.items():
    print(f'\n🔵 {plat.upper()} (N_p={res["N_p"]}, alpha={ALPHA})')
    for f in res['idf_raw']:
        print(f'   {f:15s}: IDF_raw={res["idf_raw"][f]:7.4f} | IDF_norm={res["idf_norm"][f]:.4f}')

## 5️⃣ Rumus Akhir: Engagement Score per Platform (V2)

In [ ]:
print('='*75)
print('🧮 RUMUS AKHIR ENGAGEMENT SCORE V2 PER PLATFORM')
print(f'   IDF_{{p,f}} = log2((N_p + {ALPHA}) / (DF_{{p,f}} + {ALPHA}))')
print('   ES = Σ (nilai_faktor × IDF_norm)')
print('='*75)

for plat, res in results_v2.items():
    w = res['idf_norm']
    idf = res['idf_raw']
    N_p = res['N_p']
    st = df_stats[plat]
    
    print(f'\n📌 {plat.upper()} (N_p={N_p})')
    terms = []
    for f, ww in w.items():
        df_pf = st.get(f, 0)
        idf_v = idf[f]
        terms.append(f'{f}×{ww:.4f}')
        print(f'   DF[{f}]={df_pf} → IDF_raw={idf_v:.4f} → IDF_norm={ww:.4f}')
    print(f'   ➤ ES_{plat[:2]} = {" + ".join(terms)}')

print('\n✅ V2: Tidak ada risiko division-by-zero berkat smoothing alpha!')

## 6️⃣ Visualisasi IDF Weights per Platform

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, (plat, res) in enumerate(results_v2.items()):
    ax = axes[i]
    factors = list(res['idf_raw'].keys())
    raw_vals = [res['idf_raw'][f] for f in factors]
    norm_vals = [res['idf_norm'][f] for f in factors]
    
    x = np.arange(len(factors))
    w = 0.35
    ax.bar(x - w/2, raw_vals, w, label='IDF Raw', color=COLORS[plat], alpha=0.7)
    ax.bar(x + w/2, norm_vals, w, label='IDF Norm', color='gold', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(factors, rotation=20, fontsize=8)
    ax.set_title(f'{plat.capitalize()} — IDF Weights V2', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)
    ax.axhline(0, color='red', lw=0.8, linestyle='--')

plt.suptitle(f'V2 — IDF Weights per Platform (α={ALPHA})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_v2_idf_weights.png', dpi=150, bbox_inches='tight')
plt.show()

## 7️⃣ Distribusi & Perbandingan Engagement Score V2

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, (plat, res) in enumerate(results_v2.items()):
    ax = axes[i]
    scores = res['df']['engagement_score_v2'].dropna()
    ax.hist(scores, bins=30, color=COLORS[plat], alpha=0.75, edgecolor='white')
    ax.axvline(scores.median(), color='yellow', lw=2, linestyle='--', label=f'Median={scores.median():.1f}')
    ax.axvline(scores.mean(), color='red', lw=2, label=f'Mean={scores.mean():.1f}')
    ax.set_title(f'{plat.capitalize()} — Score V2', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('V2 — Distribusi Engagement Score per Platform', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_v2_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 8️⃣ Sensitivity Analysis: Pengaruh Alpha terhadap IDF Weights

In [ ]:
# Analisis sensitivitas alpha: bagaimana perubahan alpha mempengaruhi IDF
alphas = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, (plat, df) in enumerate(dfs.items()):
    ax = axes[i]
    eng = [c for c in df.columns if c not in ['post_id','platform']]
    N_p = len(df)
    
    for col in eng:
        DF_pf = (df[col] > 0).sum()
        idf_by_alpha = [np.log2((N_p + a) / (DF_pf + a)) for a in alphas]
        ax.plot(alphas, idf_by_alpha, marker='o', label=col, linewidth=2)
    
    ax.set_title(f'{plat.capitalize()} — Sensitivitas Alpha', fontweight='bold')
    ax.set_xlabel('Alpha (α)')
    ax.set_ylabel('IDF Value')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)
    ax.axhline(0, color='red', lw=0.8, linestyle='--')

plt.suptitle('V2 — Pengaruh Alpha (α) terhadap IDF Weight per Faktor', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_v2_alpha_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n💡 Semakin besar α, IDF semakin stabil & mendekati nol → smoothing lebih kuat')

## 9️⃣ Ringkasan & Top 10 per Platform

In [ ]:
# Summary table
summary = []
for plat, res in results_v2.items():
    s = res['df']['engagement_score_v2']
    summary.append({'Platform': plat.capitalize(),
                    'N Posts': len(res['df']),
                    'Mean': round(s.mean(),2),
                    'Median': round(s.median(),2),
                    'Max': round(s.max(),2),
                    'Std': round(s.std(),2)})

df_sum = pd.DataFrame(summary).set_index('Platform')
print('\n📊 RINGKASAN ENGAGEMENT SCORE V2')
print(df_sum.to_string())

# Bar chart perbandingan mean score
fig, ax = plt.subplots(figsize=(12, 5))
plats = [s['Platform'] for s in summary]
means = [s['Mean'] for s in summary]
clrs = [COLORS[p.lower()] for p in plats]
bars = ax.bar(plats, means, color=clrs, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, means):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01*max(means),
            f'{val:,.1f}', ha='center', fontweight='bold')
ax.set_title(f'V2 — Mean Engagement Score per Platform (α={ALPHA})', fontsize=13, fontweight='bold')
ax.set_ylabel('Mean Engagement Score')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('eda_v2_mean_score.png', dpi=150, bbox_inches='tight')
plt.show()

# Top 10
print('\n🏆 TOP 10 POSTS — ENGAGEMENT SCORE V2')
for plat, res in results_v2.items():
    df = res['df']
    top = df.nlargest(5, 'engagement_score_v2')
    eng = [c for c in df.columns if c not in ['platform']]
    print(f'\n🥇 {plat.upper()}')
    print(top[eng].to_string(index=False))